In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm import camels, plotting, hpm, nn, graph, diagnostics, objectives, egd
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, ScaleConditionedCNN
from jaxpm.objectives import ParticleLoss, FieldLoss

print(jax.default_backend())

/global/common/software/des/athomsen/flatiron/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


gpu


# configuration

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
# mesh_per_dim = 2 * parts_per_dim

# parts_per_dim = 128
# mesh_per_dim = parts_per_dim

# parts_per_dim = None
# mesh_per_dim = 256

mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# CAMELS

In [4]:
CODE = "SIMBA"
# CODE = "ASTRID"
# CODE = "IllustrisTNG"

train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

vali_dict = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

# for EGD
vali_dict_nbody = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=[-1, -1],
    return_hydro=False,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)


vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

# general
cosmo = vali_dict["cosmo"]
scales = vali_dict["scales"]

# particles
dm_poss = vali_dict["dm_poss"]
dm_vels = vali_dict["dm_vels"]

gas_poss = vali_dict["gas_poss"]
gas_vels = vali_dict["gas_vels"]

# masses
dm_mass = cosmo.Omega_c / (cosmo.Omega_b + cosmo.Omega_c)
gas_mass = 1 - dm_mass

# fields

# rhos_dm = vcic_paint(jnp.zeros(mesh_shape), dm_poss, dm_mass)
rhos_dm = vcic_paint(jnp.zeros(mesh_shape), dm_poss, 1)
deltas_dm = rhos_dm/rhos_dm.mean() - 1

# rhos_gas = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
rhos_gas = vcic_paint(jnp.zeros(mesh_shape), gas_poss, 1)
deltas_gas = rhos_gas/rhos_gas.mean() - 1

# power spectrum
_, cls_dm = objectives.vpower_spectrum(deltas_dm)
_, cls_gas = objectives.vpower_spectrum(deltas_gas)

Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=64,mesh=64.h5
Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_1/parts=64,mesh=64.h5
Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA_DM/CV/CV_1/parts=64,mesh=64,i=[-1, -1].h5


In [5]:
dt0 = 0.01

def solve_ode(y0, t0, ts, gravity_model, pressure_model, training=True):
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=gravity_model, 
            pressure_model=pressure_model, 
            training=training,
            # precomputed_edges=edges,
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=t0,
            t1=ts[-1],
            dt0=dt0,
            y0=y0,
            saveat=SaveAt(ts=ts),
            max_steps=1000,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res


In [6]:
checkpoint_dir = "/global/homes/a/athomsen/flatiron/JaxPM/dev/hpm/sims/checkpoints"

# gravity correction

In [7]:
gravity_model = nn.NeuralSplineFourierFilterNNX(n_knots=8, d_latent=16, rngs=nnx.Rngs(0))

checkpoint_file = f"gravity_fourier_spline_parts={parts_per_dim},mesh={mesh_per_dim},dt={dt0}.jx"
checkpoint_file = os.path.join(checkpoint_dir, checkpoint_file)

abstract_model = nnx.eval_shape(lambda: gravity_model)
graphdef, abstract_params = nnx.split(abstract_model)

checkpointer = ocp.StandardCheckpointer()
params = checkpointer.restore(checkpoint_file, abstract_params)
gravity_model = nnx.merge(graphdef, params)

/global/common/software/des/athomsen/flatiron/lib/python3.11/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1269: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


# pressure correction

### MLP

In [8]:
pressure_model = MLP(
    d_in=5,
    d_out=1, 
    d_hidden=64, 
    n_hidden=4, 
    dropout_rate=0.0,
    rngs=nnx.Rngs(0),
    norm_type="layer",
    activation=jax.nn.swish,
)

def load_mlp_checkpoint(pressure_model, checkpoint_file):    
    checkpoint_file = os.path.join(checkpoint_dir, checkpoint_file)
    
    abstract_model = nnx.eval_shape(lambda: pressure_model)
    graphdef, abstract_params = nnx.split(abstract_model)
    
    checkpointer = ocp.StandardCheckpointer()
    params = checkpointer.restore(checkpoint_file, abstract_params)
    pressure_model = nnx.merge(graphdef, params)

    return pressure_model

# two-point evolution

In [9]:
i0 = -8
i_loss = np.arange(i0, 0, dtype=int)

checkpoint_file = f"mlp_pos_only_last_8.jx"
checkpoint_file = os.path.join(checkpoint_dir, checkpoint_file)

abstract_model = nnx.eval_shape(lambda: pressure_model)
graphdef, abstract_params = nnx.split(abstract_model)

checkpointer = ocp.StandardCheckpointer()
params = checkpointer.restore(checkpoint_file, abstract_params)
pressure_model = nnx.merge(graphdef, params)

ValueError: User-provided restore item and on-disk value metadata tree structures do not match: {'k_model': Diff(lhs=None, rhs={'linear_a1': {'bias': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(16,))}, 'kernel': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(1, 16))}}, 'linear_a2': {'bias': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(16,))}, 'kernel': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(16, 16))}}, 'linear_k': {'bias': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(7,))}, 'kernel': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(16, 7))}}, 'linear_w': {'bias': {'value': ValueMetadataEntry(value_type='jax.Array', skip_deserialize=False, write_shape=(9,))}, 'kernel': {'value': ValueMetadataEntry(value_type='jax.Array',

In [ ]:
# particle mesh
y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
t0 = scales[i0]
ts = scales[i_loss]

pm_res = solve_ode(y0, t0, ts, None, None)
pm_dm_poss, pm_dm_vels, pm_gas_poss, pm_gas_vels = pm_res

nn_res = solve_ode(y0, t0, ts, gravity_model, pressure_model)
nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res

def make_deltas(poss):
    rhos = vcic_paint(jnp.zeros(mesh_shape), poss, 1)
    deltas = rhos/rhos.mean() - 1
    return deltas

pm_deltas_gas = make_deltas(pm_gas_poss)
nn_deltas_gas = make_deltas(nn_gas_poss)

# CAMELS
ref_gas_poss, ref_gas_vels, ref_deltas_gas = gas_poss[i_loss], gas_vels[i_loss], deltas_gas[i_loss]

In [ ]:
k, ref_pk_gas = objectives.vpower_spectrum(ref_deltas_gas)
_, pm_pk_gas = objectives.vpower_spectrum(pm_deltas_gas)
_, nn_pk_gas = objectives.vpower_spectrum(nn_deltas_gas)

_, ref_xpk_gas = objectives.vcross_correlation(ref_deltas_gas, ref_deltas_gas)
_, pm_xpk_gas = objectives.vcross_correlation(pm_deltas_gas, ref_deltas_gas)
_, nn_xpk_gas = objectives.vcross_correlation(nn_deltas_gas, ref_deltas_gas)

k = k[0]

In [ ]:
pos_loss = ParticleLoss(
    mesh_per_dim,
    w_pos=1.0,
    w_vel=0.0,
    w_cls=0.0,
    w_cross=0.0,
    w_snapshot=0.0,
    cutoff_quantile=0.99,
)

nn_loss = pos_loss(ref_gas_poss, None, nn_gas_poss, snapshot_mean=False)
pm_loss = pos_loss(ref_gas_poss, None, pm_gas_poss, snapshot_mean=False)

In [ ]:
# fig, ax = plt.subplots(ncols=3, figsize=(3*4,4))
fig, ax = plt.subplots(ncols=2, figsize=(2*4,4))

cmap = "magma"
n_snapshots = len(i_loss)

colors = sns.color_palette(cmap, n_snapshots)
# for i in range(0, n_snapshots, 2):
for i in range(n_snapshots):
    ax[0].plot(k, nn_pk_gas[i]/ref_pk_gas[i] - 1, color=colors[i])
    ax[0].plot(k, pm_pk_gas[i]/ref_pk_gas[i] - 1, color=colors[i], linestyle=":")

    ax[1].plot(k, nn_xpk_gas[i]/jnp.sqrt(nn_pk_gas[i]*ref_pk_gas[i]) - 1, color=colors[i])
    ax[1].plot(k, pm_xpk_gas[i]/jnp.sqrt(pm_pk_gas[i]*ref_pk_gas[i]) - 1, color=colors[i], linestyle=":")
    
    # ax[2].plot(ts, pm_loss)
    # ax[2].plot(ts, nn_loss)

ax[0].set(xscale="log", yscale="linear")
ax[1].set(xscale="log", yscale="linear")

# make a scalar mappable for the colorbar
cmap = mpl.cm.get_cmap(cmap)
norm = mpl.colors.Normalize(vmin=ts.min(), vmax=ts.max())
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# add colorbar below both plots
cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.2)
cbar.set_label("scale factor")

cbar.set_ticks(ts)
cbar.set_ticklabels([f"{t:.2f}" for t in ts])

# two-point final

In [ ]:
checkpoint_file = "mlp_cls_0_to_last_4.jx"
pressure_model = load_mlp_checkpoint(pressure_model, checkpoint_file)

In [ ]:
i0 = 0
i_loss = np.arange(-4, 0, dtype=int)

# particle mesh
y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
t0 = scales[i0]
ts = scales[i_loss]

pm_res = solve_ode(y0, t0, ts, None, None)
pm_dm_poss, pm_dm_vels, pm_gas_poss, pm_gas_vels = pm_res

nn_res = solve_ode(y0, t0, ts, gravity_model, pressure_model)
nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res

def make_deltas(poss):
    rhos = vcic_paint(jnp.zeros(mesh_shape), poss, 1)
    deltas = rhos/rhos.mean() - 1
    return deltas, rhos

pm_deltas_dm, pm_rhos_dm = make_deltas(pm_dm_poss)
nn_deltas_dm, nn_rhos_dm = make_deltas(nn_dm_poss)

pm_deltas_gas, pm_rhos_gas = make_deltas(pm_gas_poss)
nn_deltas_gas, nn_rhos_gas = make_deltas(nn_gas_poss)

# CAMELS
ref_gas_poss, ref_gas_vels, ref_deltas_gas, ref_rhos_gas = gas_poss[i_loss], gas_vels[i_loss], deltas_gas[i_loss], rhos_gas[i_loss]
ref_dm_poss, ref_dm_vels, ref_deltas_dm, ref_rhos_dm = dm_poss[i_loss], dm_vels[i_loss], deltas_dm[i_loss], rhos_dm[i_loss]

# two-point 
k, ref_pk_gas = objectives.vpower_spectrum(ref_deltas_gas)
_, pm_pk_gas = objectives.vpower_spectrum(pm_deltas_gas)
_, nn_pk_gas = objectives.vpower_spectrum(nn_deltas_gas)

i1 = -1

### EGD

In [ ]:
egd_params = [
    np.array(0.79741, dtype=np.float32),
    np.array(7.9030437, dtype=np.float32),
    np.array(0.51323, dtype=np.float32)
]

egd_rho_dm, egd_rho_gas, egd_rho_tot = egd.apply_egd_correction(egd_params, vali_dict_nbody["dm_poss"][i1], mesh_shape, cosmo)

# re-normalize the gas particles to weight 1 instead of Ob/Om
# egd_rho_gas *= mesh_per_dim**3 / egd_rho_gas.sum()
# egd_rho_tot *= mesh_per_dim**3 / egd_rho_tot.sum()

In [ ]:
egd_rho_tot.sum()

In [ ]:
pm_rho_tot

In [ ]:
ref_rho_tot.sum()

In [ ]:
ref_rho_tot.sum()

In [ ]:
egd_rho_tot_plot.sum()

In [ ]:
gas_mass + dm_mass

In [ ]:
pm_rho_dm = dm_mass * pm_rhos_dm[i1]
pm_rho_gas = gas_mass * pm_rhos_gas[i1]
pm_rho_tot = pm_rho_dm + pm_rho_gas

In [ ]:
pm_rho_tot.sum()

In [ ]:
egd_rho_tot

In [ ]:
pm_rho_gas.shape

In [ ]:
pm_rho_gas.sum()

In [ ]:
64**3*gas_mass

In [ ]:
# def rho_transform(rhos, i):
#     rho = rhos[i]
#     rho_plot = rho.sum(axis=0)
#     return rho_plot

# pm_rho_gas = rho_transform(gas_mass * pm_rhos_gas, i1)
# nn_rho_gas = rho_transform(gas_mass * nn_rhos_gas, i1)
# ref_rho_gas = rho_transform(gas_mass * ref_rhos_gas, i1)

# pm_rho_tot = rho_transform(gas_mass * pm_rho_gas + dm_mass * pm_rhos_dm, i1)
# nn_rho_tot = rho_transform(gas_mass * nn_rho_gas + dm_mass * nn_rhos_dm, i1)
# ref_rho_tot = rho_transform(gas_mass * ref_rho_gas + dm_mass * ref_rhos_dm, i1)

# egd_rho_gas_plot = (gas_mass * egd_rho_gas).sum(axis=0)
# egd_rho_tot_plot = egd_rho_tot.sum(axis=0)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec

# Tunable layout parameters
subplot_size = 4        # size of each square subplot in inches
hspace = 0.1            # vertical spacing between rows
wspace = 0.1           # horizontal spacing between columns
cbar_width = 0.1       # width of vertical colorbar as fraction of figure
cbar_pad = 0.02         # padding between subplot and colorbar

nrows, ncols = 2, 4
fig = plt.figure(figsize=(ncols*subplot_size, nrows*subplot_size))
gs = GridSpec(nrows, ncols, figure=fig, hspace=hspace, wspace=wspace)

# Helper function to plot a row of fields with a shared vertical colorbar
def plot_row(fields, row_index, cmap, percentile=(1,99), ylabel=None):
    # Stack fields for computing vmin/vmax
    vmin = jnp.percentile(jnp.stack(fields), percentile[0])
    vmax = jnp.percentile(jnp.stack(fields), percentile[1])
    # vmin = 1e1
    # vmin = 1e3
    print(f"vlim = [{vmin:.2f}, {vmax:.2f}]")
    
    norm = LogNorm(vmin=vmin, vmax=vmax)

    axs = []
    for j, field in enumerate(fields):
        ax = fig.add_subplot(gs[row_index, j])
        im = ax.imshow(field, norm=norm, cmap=cmap)
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0 and ylabel is not None:
            ax.set_ylabel(ylabel, fontsize=20)
        axs.append(ax)

    # Colorbar: aligned with subplot height
    cbar_ax = fig.add_axes([
        axs[-1].get_position().x1 + cbar_pad,
        axs[0].get_position().y0,
        cbar_width * subplot_size / fig.get_size_inches()[0],
        axs[0].get_position().height
    ])
    fig.colorbar(im, cax=cbar_ax, orientation="vertical")
    return axs

# Top row: total density
fields_tot = [ref_rho_tot, pm_rho_tot, nn_rho_tot, egd_rho_tot_plot]
axs_top = plot_row(fields_tot, 0, cmap="mako", ylabel=r"$\rho_\text{tot} = \rho_\text{dm} + \rho_\text{gas}$")

# Bottom row: gas density
fields_gas = [ref_rho_gas, pm_rho_gas, nn_rho_gas, egd_rho_gas_plot]
axs_bottom = plot_row(fields_gas, 1, cmap="rocket", ylabel=r"$\rho_\text{gas}$")

# Column titles
titles = ["CAMELS", "JaxPM", "JaxHPM", "EGD"]
for ax, title in zip(axs_top, titles):
    ax.set_title(title, fontsize=20)

fig.savefig("plots/final_fields.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# from matplotlib.colors import LogNorm

# def rho_transform(rhos, i):
#     rho = rhos[i]
#     rho_plot = rho.sum(axis=0)
#     return rho_plot

# pm_rho_gas = rho_transform(gas_mass * pm_rhos_gas, i1)
# nn_rho_gas = rho_transform(gas_mass * nn_rhos_gas, i1)
# ref_rho_gas = rho_transform(gas_mass * ref_rhos_gas, i1)

# pm_rho_tot = rho_transform(gas_mass * pm_rho_gas + dm_mass * pm_rhos_dm, i1)
# nn_rho_tot = rho_transform(gas_mass * nn_rho_gas + dm_mass * nn_rhos_dm, i1)
# ref_rho_tot = rho_transform(gas_mass * ref_rho_gas + dm_mass * ref_rhos_dm, i1)

# egd_rho_gas_plot = (gas_mass * egd_rho_gas).sum(axis=0)
# egd_rho_tot_plot = egd_rho_tot.sum(axis=0)

# fig, ax = plt.subplots(ncols=4, nrows=2, figsize=(4*4, 2*4-1.5))

# cmap = "viridis"
# rho_tot = jnp.stack([pm_rho_tot, nn_rho_tot, ref_rho_tot, egd_rho_tot_plot])
# vmin_tot = jnp.percentile(rho_tot, 1)
# vmax_tot = jnp.percentile(rho_tot, 99)
# norm = LogNorm(vmin=vmin_tot, vmax=vmax_tot)
# im = ax[0,0].imshow(ref_rho_tot, norm=norm, cmap=cmap)
# im = ax[0,1].imshow(pm_rho_tot, norm=norm, cmap=cmap)
# im = ax[0,2].imshow(nn_rho_tot, norm=norm, cmap=cmap)
# im = ax[0,3].imshow(egd_rho_tot_plot, norm=norm, cmap=cmap)
# fig.colorbar(im, ax=ax[0,:], orientation="vertical", shrink=0.91, aspect=10, pad=0.02)

# cmap = "magma"
# rho_gas = jnp.stack([pm_rho_gas, nn_rho_gas, ref_rho_gas, egd_rho_gas_plot])
# vmin_gas = jnp.percentile(rho_gas, 1)
# vmax_gas = jnp.percentile(rho_gas, 99)
# norm = LogNorm(vmin=vmin_gas, vmax=vmax_gas)
# im = ax[1,0].imshow(ref_rho_gas, norm=norm, cmap=cmap)
# im = ax[1,1].imshow(pm_rho_gas, norm=norm, cmap=cmap)
# im = ax[1,2].imshow(nn_rho_gas, norm=norm, cmap=cmap)
# im = ax[1,3].imshow(egd_rho_gas_plot, norm=norm, cmap=cmap)
# fig.colorbar(im, ax=ax[1,:], orientation="vertical", shrink=0.91, aspect=10, pad=0.02)

# ax[0,0].set_title("CAMELS", fontsize=18)
# ax[0,1].set_title("JaxPM", fontsize=18)
# ax[0,2].set_title("JaxHPM", fontsize=18)
# ax[0,3].set_title("EGD", fontsize=18)

# ax[0,0].set_ylabel(r"$\rho_\text{tot}$", fontsize=18)
# ax[1,0].set_ylabel(r"$\rho_\text{gas}$", fontsize=18)

# for i in range(ax.shape[0]):
#     for j in range(ax.shape[1]):
#         ax[i,j].set_xticks([])
#         ax[i,j].set_yticks([])

# fig.savefig("plots/final_fields.pdf", bbox_inches="tight")

In [ ]:
# i1 = -1

# def rho_transform(rhos, i):
#     rho = rhos[i]
#     rho_plot = jnp.log(rho.sum(axis=0))
#     return rho_plot

# pm_rho_gas = rho_transform(gas_mass * pm_rhos_gas, i1)
# nn_rho_gas = rho_transform(gas_mass * nn_rhos_gas, i1)
# ref_rho_gas = rho_transform(gas_mass * ref_rhos_gas, i1)

# pm_rho_tot = rho_transform(gas_mass * pm_rho_gas + dm_mass * pm_rhos_dm, i1)
# nn_rho_tot = rho_transform(gas_mass * nn_rho_gas + dm_mass * nn_rhos_dm, i1)
# ref_rho_tot = rho_transform(gas_mass * ref_rho_gas + dm_mass * ref_rhos_dm, i1)

# egd_rho_gas_plot = jnp.log(egd_rho_gas.sum(axis=0))
# egd_rho_tot_plot = jnp.log(egd_rho_tot.sum(axis=0))

# fig, ax = plt.subplots(ncols=4, nrows=2, figsize=(4*4, 2*4))

# cmap = "viridis"
# rho_tot = jnp.stack([pm_rho_tot, nn_rho_tot, ref_rho_tot, egd_rho_tot_plot])
# vmin_tot = jnp.percentile(rho_tot, 1)
# vmax_tot = jnp.percentile(rho_tot, 99)
# im = ax[0,0].imshow(ref_rho_tot, vmin=vmin_tot, vmax=vmax_tot, cmap=cmap)
# im = ax[0,1].imshow(pm_rho_tot, vmin=vmin_tot, vmax=vmax_tot, cmap=cmap)
# im = ax[0,2].imshow(nn_rho_tot, vmin=vmin_tot, vmax=vmax_tot, cmap=cmap)
# im = ax[0,3].imshow(egd_rho_tot_plot, vmin=vmin_tot, vmax=vmax_tot, cmap=cmap)
# fig.colorbar(im, ax=ax[0,:], orientation="vertical", shrink=0.7, aspect=10, label="log(sum(field))")

# cmap = "magma"
# rho_gas = jnp.stack([pm_rho_gas, nn_rho_gas, ref_rho_gas, egd_rho_gas_plot])
# vmin_gas = jnp.percentile(rho_gas, 1)
# vmax_gas = jnp.percentile(rho_gas, 99)

# im = ax[1,0].imshow(ref_rho_gas, vmin=vmin_gas, vmax=vmax_gas, cmap=cmap)
# im = ax[1,1].imshow(pm_rho_gas, vmin=vmin_gas, vmax=vmax_gas, cmap=cmap)
# im = ax[1,2].imshow(nn_rho_gas, vmin=vmin_gas, vmax=vmax_gas, cmap=cmap)
# im = ax[1,3].imshow(egd_rho_gas_plot, vmin=vmin, vmax=vmax, cmap=cmap)
# fig.colorbar(im, ax=ax[1,:], orientation="vertical", shrink=0.7, aspect=10, label="log(sum(field))")

# ax[0,0].set_title("CAMELS", fontsize=18)
# ax[0,1].set_title("PM", fontsize=18)
# ax[0,2].set_title("HPM", fontsize=18)
# ax[0,3].set_title("EGD", fontsize=18)

# ax[0,0].set_ylabel("total matter", fontsize=18)
# ax[1,0].set_ylabel("gas", fontsize=18)

# for i in range(ax.shape[0]):
#     for j in range(ax.shape[1]):
#         ax[i,j].set_xticks([])
#         ax[i,j].set_yticks([])



In [ ]:
# i1 = -1

# pm_rho_gas = pm_rhos_gas[i1]
# nn_rho_gas = nn_rhos_gas[i1]
# ref_rho_gas = ref_rhos_gas[i1]

# pm_rho_gas = jnp.log(pm_rho_gas.sum(axis=0))
# nn_rho_gas = jnp.log(nn_rho_gas.sum(axis=0))
# ref_rho_gas = jnp.log(ref_rho_gas.sum(axis=0))
# egd_rho_gas = jnp.log(egd_rho_gas.sum(axis=0))

# fig, ax = plt.subplots(ncols=4, figsize=(4*4, 4))

# cmap = "magma"
# vmin = jnp.percentile(jnp.stack([pm_rho_gas, nn_rho_gas, ref_rho_gas]), 1)
# vmax = jnp.percentile(jnp.stack([pm_rho_gas, nn_rho_gas, ref_rho_gas]), 99)

# im = ax[0].imshow(ref_rho_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# im = ax[1].imshow(pm_rho_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# im = ax[2].imshow(nn_rho_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# im = ax[3].imshow(egd_rho_gas, vmin=vmin, vmax=vmax, cmap=cmap)

# ax[0].set(title="CAMELS")
# ax[1].set(title="PM")
# ax[2].set(title="HPM")
# ax[3].set(title="EGD")

# for i in range(4):
#     ax[i].set_xticks([])
#     ax[i].set_yticks([])

# fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.7, aspect=20, label="log(sum(field))")

In [ ]:
# i1 = -1

# pm_delta_gas = pm_deltas_gas[i1]
# nn_delta_gas = nn_deltas_gas[i1]
# ref_delta_gas = ref_deltas_gas[i1]

# pm_delta_gas = pm_delta_gas.sum(axis=0)
# nn_delta_gas = nn_delta_gas.sum(axis=0)
# ref_delta_gas = ref_delta_gas.sum(axis=0)

# s = jnp.percentile(jnp.abs(jnp.stack([pm_delta_gas, nn_delta_gas, ref_delta_gas])), 90)
# pm_delta_gas = jnp.arcsinh(pm_delta_gas / s)
# nn_delta_gas = jnp.arcsinh(nn_delta_gas / s)
# ref_delta_gas = jnp.arcsinh(ref_delta_gas / s)




# fig, ax = plt.subplots(ncols=4, figsize=(4 * 4, 4))

# cmap = "magma"
# vmin = jnp.percentile(jnp.stack([pm_delta_gas, nn_delta_gas, ref_delta_gas]), 1)
# vmax = jnp.percentile(jnp.stack([pm_delta_gas, nn_delta_gas, ref_delta_gas]), 99)

# im = ax[0].imshow(ref_delta_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# im = ax[1].imshow(pm_delta_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# im = ax[2].imshow(nn_delta_gas, vmin=vmin, vmax=vmax, cmap=cmap)
# # im = ax[3].imshow(egd_delta_gas, vmin=vmin, vmax=vmax, cmap=cmap)

# ax[0].set(title="CAMELS")
# ax[1].set(title="PM")
# ax[2].set(title="HPM")

# for i in range(3):
#     ax[i].set_xticks([])
#     ax[i].set_yticks([])

# fig.colorbar(im, ax=ax[:3], orientation="horizontal", shrink=0.8, aspect=50, label="log(sum(field))")


### difference with respect to CAMELS

In [ ]:
# i1 = -1

# diff_pm = jnp.log(pm_rhos_gas[i1].sum(axis=0)) - jnp.log(ref_rhos_gas[i1].sum(axis=0))
# diff_nn = jnp.log(nn_rhos_gas[i1].sum(axis=0)) - jnp.log(ref_rhos_gas[i1].sum(axis=0))

# vmin = jnp.minimum(diff_pm.min(), diff_nn.min())
# vmax = jnp.maximum(diff_pm.max(), diff_nn.max())

# fig, ax = plt.subplots(ncols=2)

# ax[0].imshow(diff_pm, vmin=vmin, vmax=vmax)
# ax[1].imshow(diff_nn, vmin=vmin, vmax=vmax)

In [ ]:
# i1 = -1

# diff_pm = jnp.arcsinh((pm_deltas_gas[i1] - ref_deltas_gas[i1]).sum(axis=0))
# diff_nn = jnp.arcsinh((nn_deltas_gas[i1] - ref_deltas_gas[i1]).sum(axis=0))
# # diff_pm = jnp.arcsinh((pm_rhos_gas[i1] - ref_rhos_gas[i1]).sum(axis=0))
# # diff_nn = jnp.arcsinh((nn_rhos_gas[i1] - ref_rhos_gas[i1]).sum(axis=0))
# # diff_pm = (pm_rhos_gas[i1] - ref_rhos_gas[i1]).sum(axis=0)
# # diff_nn = (nn_rhos_gas[i1] - ref_rhos_gas[i1]).sum(axis=0)

# vmin = jnp.minimum(diff_pm.min(), diff_nn.min())
# vmax = jnp.maximum(diff_pm.max(), diff_nn.max())

# fig, ax = plt.subplots(ncols=2)

# ax[0].imshow(diff_pm, vmin=vmin, vmax=vmax)
# ax[1].imshow(diff_nn, vmin=vmin, vmax=vmax)

# fig, ax = plt.subplots()

# ax.hist(diff_pm.flatten())

# pressure comparison